[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（四步框架自检器 / 沟通三铁律打分器 / 好坏回答结构对比）

目标：把"结构化表达"从一堆经验之谈，变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检** —— 确认 Python / numpy 可用（本课全程不需要 GPU、不需要联网）
2. **四步框架自检器** —— 给一段"你实际做了什么"的动作序列，自动指出漏了哪步、哪两步顺序反了
3. **四步框架完整度打分器** —— 把"缺步"和"顺序违规"合成一个 0-1 分
4. **沟通三铁律打分器** —— 给一段回答的三个布尔标记打分，找出最该补的一条
5. **"好回答 vs 坏回答"结构分析与打分器** —— 把自然语言回答表示成一串"言语行为标签"，量化两者的差距
6. **10 秒结构预算器** —— 把"先给结构再展开"这句话变成一个可判定的字数上限

> 心智模型：**这个环节考的不是"你知道多少"，是"你的推理过程能不能被别人看见"。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math, random
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'argsort')

print('\n环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 四步框架自检器

框架本身是一个有序清单。自检器要回答两件事：
**（a）你漏了哪一步？（b）哪两步的顺序反了？**

顺序违规的定义：设框架里 `a` 应在 `b` 之前，但你**首次**做 `b` 的时刻早于首次做 `a` —— 记一次违规 `(b, a)`。

In [ ]:
FRAMEWORK_STEPS = [
    ('clarify',     '① 澄清',       '确认目标、边界、约束、成功标准'),
    ('decompose',   '② 分解',       '把问题拆成互相独立、可逐个处理的子问题'),
    ('hypothesize', '③ 假设与验证', '提出一个可证伪的假设，说清怎么验证'),
    ('converge',    '④ 收敛与权衡', '给出结论，显式说清放弃了什么、为什么'),
]
FRANK = {k: i for i, (k, _, _) in enumerate(FRAMEWORK_STEPS)}

def check_framework(trace):
    """trace: 你实际做的动作序列（step key 的列表，可重复）。
       返回 (missing, inversions)：缺失的步骤、以及顺序违规对 (先做的, 本该更早的)。"""
    missing = [k for k, _, _ in FRAMEWORK_STEPS if k not in trace]
    first = {}
    for pos, t in enumerate(trace):
        first.setdefault(t, pos)
    inversions = set()
    present = [k for k in FRANK if k in first]
    for a in present:
        for b in present:
            if FRANK[a] < FRANK[b] and first[a] > first[b]:
                inversions.add((b, a))
    return missing, sorted(inversions, key=lambda p: (FRANK[p[0]], FRANK[p[1]]))

for k, name, what in FRAMEWORK_STEPS:
    print(f'{name:<10} {what}')

In [ ]:
# —— 三种典型表现 ——
good  = ['clarify', 'decompose', 'hypothesize', 'converge']    # 规范顺序
rush  = ['converge']                                           # 一上来就给结论，没有推理过程
mixed = ['clarify', 'converge', 'decompose', 'hypothesize']    # 先给了结论，才回头分解验证

m1, i1 = check_framework(good)
assert m1 == [] and i1 == []

m2, i2 = check_framework(rush)
assert set(m2) == {'clarify', 'decompose', 'hypothesize'}, m2
assert i2 == [], '只做了 converge 一步，没有别的步骤可比较顺序'

m3, i3 = check_framework(mixed)
assert m3 == []
assert i3 == [('converge', 'decompose'), ('converge', 'hypothesize')], i3
assert len(i3) == 2

print('good  -> 缺失', m1, '违规', i1)
print('rush  -> 缺失', m2, '  <- 丢掉澄清/分解/假设验证三步的分，只剩一个孤零零的结论')
print('mixed -> 违规', i3, '  <- 结论说在了分解与验证之前')
print('\n自检器就位：把模拟面试的录像回放一遍，把动作打成 trace 喂进来。')

## 2 · 四步框架完整度打分器

把"缺步"和"顺序违规"合成一个 0-1 分：`completeness = (4 - 缺步数) / 4`，
`penalty = 0.15 × 违规对数`，`score = max(0, completeness - penalty)`。

In [ ]:
def framework_score(trace):
    """结合步骤完整度与顺序违规惩罚给出 0-1 分。"""
    missing, inversions = check_framework(trace)
    completeness = (4 - len(missing)) / 4
    penalty = 0.15 * len(inversions)
    return max(0.0, completeness - penalty)

s_good, s_rush, s_mixed = framework_score(good), framework_score(rush), framework_score(mixed)
assert abs(s_good - 1.0) < 1e-9, s_good
assert abs(s_rush - 0.25) < 1e-9, s_rush
assert abs(s_mixed - 0.7) < 1e-9, s_mixed

for name, s in [('good', s_good), ('rush', s_rush), ('mixed', s_mixed)]:
    print(f'{name:<6} -> {s:.2f}')
print('\n完整但顺序反了（mixed）仍然好于严重缺步（rush）——')
print('说明"至少把四步都提到"，比"顺序完美但漏了大半"更重要。')

## 3 · 沟通三铁律打分器

三律：结论先行 / 显式说假设 / 主动暴露不确定性，各占三分之一权重。

In [ ]:
LAWS = [
    ('conclusion_first',     '结论先行'),
    ('explicit_assumptions', '显式说假设'),
    ('surfaced_uncertainty', '主动暴露不确定性'),
]

def score_communication(flags):
    """flags: dict{law_key: bool}。返回 (score 0-1, 未达标的律 list)。"""
    keys = [k for k, _ in LAWS]
    hit = sum(1 for k in keys if flags.get(k, False))
    missed = [k for k in keys if not flags.get(k, False)]
    return hit / len(keys), missed

GOOD_FLAGS = dict(conclusion_first=True, explicit_assumptions=True, surfaced_uncertainty=True)
BAD_FLAGS = dict(conclusion_first=False, explicit_assumptions=False, surfaced_uncertainty=True)

sg, mg = score_communication(GOOD_FLAGS)
sb, mb = score_communication(BAD_FLAGS)
assert sg == 1.0 and mg == []
assert abs(sb - 1 / 3) < 1e-9, sb
assert mb == ['conclusion_first', 'explicit_assumptions'], mb

print(f'好回答: 分数={sg:.2f}  未达标={mg}')
print(f'坏回答: 分数={sb:.2f}  未达标={mb}')
print('\n坏回答唯一做对的是"暴露了不确定性"，但另外两条全丢，分数只剩三分之一。')

## 4 · "好回答 vs 坏回答"结构分析与打分

把一段回答表示成一串"言语行为标签"（发言时依次做的事）：
`conclusion`（给结论）/ `clarify_q`（提澄清问题）/ `decompose`（分解）/ `assumption`（说假设）/
`evidence`（给证据）/ `tradeoff`（说权衡）/ `uncertainty`（说不确定）/ `filler`（口水话）/ `silence`（沉默）。

In [ ]:
def analyze_answer(tags):
    """tags: 言语行为标签的有序列表。返回结构分析字典。"""
    n = len(tags)
    leads = n > 0 and tags[0] == 'conclusion'
    has_assumption = 'assumption' in tags
    has_uncertainty = 'uncertainty' in tags
    has_tradeoff = 'tradeoff' in tags
    filler = tags.count('filler') + tags.count('silence')
    filler_ratio = filler / n if n else 0.0
    # 四步框架里，哪几步在这段回答里被真正提到过
    mapping = {'clarify_q': 'clarify', 'decompose': 'decompose',
               'assumption': 'hypothesize', 'tradeoff': 'converge'}
    covered = {mapping[t] for t in tags if t in mapping}
    return dict(leads_with_conclusion=leads, has_assumption=has_assumption,
                has_uncertainty=has_uncertainty, has_tradeoff=has_tradeoff,
                filler_ratio=filler_ratio, coverage=len(covered) / 4)

def score_answer(tags):
    """把 analyze_answer 的结果合成一个 0-1 总分（下限截到 0）。
       权重：四步覆盖度 0.4，结论先行 0.2，说了假设 0.15，说了不确定性 0.15，
       口水话/沉默按比例倒扣（最多倒扣 0.3）。"""
    a = analyze_answer(tags)
    raw = (0.4 * a['coverage'] + 0.2 * a['leads_with_conclusion']
           + 0.15 * a['has_assumption'] + 0.15 * a['has_uncertainty'])
    raw -= 0.3 * a['filler_ratio']
    return max(0.0, raw)

GOOD_ANSWER = ['conclusion', 'assumption', 'clarify_q', 'decompose', 'evidence', 'tradeoff', 'uncertainty']
BAD_ANSWER = ['filler', 'filler', 'decompose', 'filler', 'conclusion', 'silence']

a_good = analyze_answer(GOOD_ANSWER)
assert a_good['coverage'] == 1.0 and a_good['leads_with_conclusion'] is True

sg2, sb2 = score_answer(GOOD_ANSWER), score_answer(BAD_ANSWER)
assert abs(sg2 - 0.9) < 1e-6, sg2
assert sb2 == 0.0, sb2

print(f'好回答标签: {GOOD_ANSWER}')
print(f'  分析: {a_good}')
print(f'  总分: {sg2:.2f}')
print(f'坏回答标签: {BAD_ANSWER}')
print(f'  总分: {sb2:.2f}')
print('\n两段回答提到的"知识维度"可能一样多，但结构打分能拉开近满分的差距。')

## 5 · 10 秒结构预算器

"先给 10 秒结构再展开"的字数上限：中文口语语速约 4-5 字/秒，10 秒约 40-50 字。
下面按 4.5 字/秒 的保守估计做判定。

In [ ]:
def opening_ok(text, cps=4.5, seconds=10):
    """判断 text 能否在 seconds 秒内、按 cps 字/秒的语速说完。"""
    return len(text) <= cps * seconds

short_text = '结论：先查数据。假设：负样本不够。接下来我按数据、模型、后处理三块看。'
long_text = '结论：' + 'x' * 30 + '。假设：' + 'y' * 30 + '。接下来我按：' + 'z' * 30 + '。'

assert opening_ok(short_text) is True, len(short_text)
assert opening_ok(long_text) is False, len(long_text)
assert opening_ok('x' * 45) is True and opening_ok('x' * 46) is False   # 45 = 4.5*10 的边界

print(f'短开场（{len(short_text)} 字，预算 45 字）-> {opening_ok(short_text)}')
print(f'长开场（{len(long_text)} 字，预算 45 字）  -> {opening_ok(long_text)}')
print('\n40-50 字大约就是"一句结论 + 一条假设 + 一句展开顺序"的长度。')

## ✏️ 练习 1：10 秒开场拼装器

实现 `opening_stub(conclusion, assumption, plan, cps=4.5, seconds=10)`，
拼出格式为 `'结论：{conclusion}。假设：{assumption}。接下来我按：{plan}。'` 的开场白，
并判断它能否在 `seconds` 秒内说完（复用 `opening_ok` 的判据）。返回 `(text, fits)`。

In [ ]:
def opening_stub(conclusion, assumption, plan, cps=4.5, seconds=10):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
text1, fits1 = opening_stub('先做数据端', '数据不足', '三步展开')
assert text1 == '结论：先做数据端。假设：数据不足。接下来我按：三步展开。', text1
assert fits1 is True, (text1, len(text1))

text2, fits2 = opening_stub('x' * 30, 'y' * 30, 'z' * 30)
assert fits2 is False, (text2, len(text2))
assert text2.startswith('结论：') and '假设：' in text2 and '接下来我按：' in text2

for t, f in [(text1, fits1), (text2, fits2)]:
    print(f'{t}\n  -> {len(t)} 字, fits={f}')
print('\n练习 1 通过：把结论/假设/计划拼成一句话，先测它是不是真的能在 10 秒内说完。')

## ✏️ 练习 2：提分收益排序（沟通三铁律版）

实现 `laws_gap(flags)`，返回 `[(铁律, 补齐它的收益), ...]`：
- 收益 = `(1/3) if not flags.get(key, False) else 0`（每条铁律权重相等）
- 按收益**降序**；收益相同时按 `LAWS` 里定义的原始顺序排列（保证结果确定）

In [ ]:
def laws_gap(flags):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gap = laws_gap(BAD_FLAGS)
names = [k for k, _ in gap]
vals = [v for _, v in gap]
assert names == ['conclusion_first', 'explicit_assumptions', 'surfaced_uncertainty'], names
assert np.allclose(vals, [1 / 3, 1 / 3, 0.0]), vals

gap_good = laws_gap(GOOD_FLAGS)
assert all(v == 0.0 for _, v in gap_good)

for k, v in gap:
    print(f'  {k:<22} 补齐可加 {v:.3f}')
print('\n练习 2 通过：坏回答该补的是"结论先行"和"显式说假设"，"暴露不确定性"这条它已经做到了。')

## ✏️ 练习 3：两段回答的胜负判定

实现 `compare_answers(tags_a, tags_b)`，复用 `score_answer`：
返回 `(winner, margin)`，`winner ∈ {'a', 'b', 'tie'}`，
当两者分数之差小于 `1e-9` 时判为 `'tie'`，否则 `margin = abs(score_a - score_b)`。

In [ ]:
def compare_answers(tags_a, tags_b):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
w, m = compare_answers(GOOD_ANSWER, BAD_ANSWER)
assert w == 'a', w
assert abs(m - 0.9) < 1e-6, m

w2, m2 = compare_answers(GOOD_ANSWER, GOOD_ANSWER)
assert w2 == 'tie' and m2 == 0.0

print(f'好回答 vs 坏回答 -> 胜者 {w}，分差 {m:.2f}')
print(f'好回答 vs 好回答 -> {w2}，分差 {m2:.2f}')
print('\n练习 3 通过：同样的知识内容，结构上的差距能拉开接近满分的差距。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def opening_stub(conclusion, assumption, plan, cps=4.5, seconds=10):
    text = f'结论：{conclusion}。假设：{assumption}。接下来我按：{plan}。'
    fits = opening_ok(text, cps=cps, seconds=seconds)
    return text, fits

In [ ]:
# 练习 2 参考答案
def laws_gap(flags):
    out = [(k, 0.0 if flags.get(k, False) else 1 / 3) for k, _ in LAWS]
    return sorted(out, key=lambda kv: -kv[1])

In [ ]:
# 练习 3 参考答案
def compare_answers(tags_a, tags_b):
    sa, sb = score_answer(tags_a), score_answer(tags_b)
    if abs(sa - sb) < 1e-9:
        return 'tie', 0.0
    return ('a' if sa > sb else 'b'), abs(sa - sb)

---
## 🧪 真实工程胶囊：四步框架 + 三铁律的现场口播模板（中英对照）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 四步框架的现场口播模板（中 / EN —— JD 是英文岗，两套都要能说）
# ══════════════════════════════════════════════════════════════════════
# 澄清 CN: 「在开始前，我想先确认几个边界条件：……」
#        EN: "Before I start, let me confirm a few boundaries: ..."
# 分解 CN: 「我把这个问题拆成三块：……」
#        EN: "I'll break this into three parts: ..."
# 假设与验证 CN: 「我先假设是……，可以通过……验证」
#            EN: "My working hypothesis is ..., which I'd verify by ..."
# 收敛与权衡 CN: 「综合来看我会选……，代价是……」
#            EN: "Weighing these, I'd go with ..., at the cost of ..."

# ══════════════════════════════════════════════════════════════════════
# B. 沟通三铁律的口播模板
# ══════════════════════════════════════════════════════════════════════
# 结论先行     CN: 「先说结论：……，下面是我的推理。」
#             EN: "My conclusion first: ..., here's the reasoning."
# 显式说假设   CN: 「这里我假设……，如果不成立，结论会变成……。」
#             EN: "I'm assuming ... here; if that doesn't hold, the answer changes to ..."
# 暴露不确定性 CN: 「这部分我不确定，把握大概六成，我会这样去验证……」
#             EN: "I'm not fully sure here, maybe 60% confident; I'd verify by ..."

# ══════════════════════════════════════════════════════════════════════
# C. 与其他板块 / 课程的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 六步编码协议（C62 模块 00）是这个框架在"产出是代码"场景下的具体化
# · ML system design 七步框架（C63 模块 00）是这个框架在"产出是架构"场景下的具体化
# · 估算（C65-01）/ 诊断（C65-02）/ 权衡（C65-03）/ 模糊需求（C65-04）/ 模拟演练（C65-05）
#   都是这四步框架在具体场景下的展开，不是四个新框架
# · 项目叙事（C61-04）用的是 STAR，讲的是"已经发生的事"，不是当场解一道新题——不要混用
'''
print(RECIPE)
for token in ['澄清', '分解', '假设', '收敛', '结论先行', 'C62 模块 00', 'C63 模块 00', 'STAR']:
    assert token in RECIPE, token
print('检查单覆盖：四步框架口播 / 三铁律口播 / 与其他课程的分工')

### 小结

- **problem-solving 考的是思维过程的可见性**：面试官买的是你的推理，不是答案本身；
  一段没有被说出来的正确判断，在评分表上等于不存在。
- **四步通用框架**（澄清 → 分解 → 假设与验证 → 收敛与权衡）是本课所有模块共用的骨架，
  且不是走一次就完事的直线——分解暴露新歧义可以跳回澄清，权衡时假设被证伪可以跳回验证。
- **沟通三铁律**：结论先行 / 显式说假设 / 主动暴露不确定性。
  其中"暴露不确定性"最反直觉——**提前承认不确定，永远比被追问戳穿要安全**。
- **"想清楚再说"和"边想边说"都不是最优解**：折中方案是先用 10 秒给一个结构性的锚点
  （结论 + 假设 + 展开顺序，字数上限约 40-50 字），再边展开边验证。
- **同样的知识内容，结构上的差距可以拉开接近满分的差距**——这是本课程存在的理由：
  这门课补的不是知识密度，是表达结构。
- **本课与相邻课程分工**：C62 六步协议 / C63 七步框架都是这个四步框架的场景特化；
  C61-04 项目叙事用 STAR，讲过去而非当场解题，不要混用。

下一站：**模块 01 · 估算题与数量级心算** —— 用 Fermi 方法把"分解"这一步做深，
学会在没有任何数据的情况下，靠锚点数字和乘法给出一个数量级正确的答案。